In [5]:
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import warnings
from sklearn.metrics import f1_score, balanced_accuracy_score, cohen_kappa_score, precision_recall_fscore_support, confusion_matrix

ROOT = Path.cwd().resolve().parent
frozen_dir = ROOT/'results'/'frozen_model'
tables_dir = ROOT/'results'/'tables'

with open(ROOT/'config.yaml','r') as file:
    config = yaml.safe_load(file)

final_model = joblib.load(frozen_dir/'final_lightgbm_model.pkl')
with open(frozen_dir/'model_metadata.json', 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_names']
classes = np.array(meta['target_classes'])

aauwss_path = ROOT/'data'/'interim'/'features_aauwss.parquet'
if not aauwss_path.exists():
    aauwss_path = ROOT/'data'/'interim'/'features_aauwss_dryrun.parquet'

df_aauwss = pd.read_parquet(aauwss_path)

stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}
df_aauwss_usable = df_aauwss[df_aauwss.usable].copy()
df_aauwss_usable['target'] = df_aauwss_usable['label'].map(stage_map)
df_aauwss_usable = df_aauwss_usable[df_aauwss_usable['target'].notna()].copy()

print('Feature compatibility check: ')
df_dreamt = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_dreamt_usable = df_dreamt[df_dreamt.usable].copy()

comp_rows = []
for col in feature_cols:
    d_mean, d_std = df_dreamt_usable[col].mean(), df_dreamt_usable[col].std()
    a_mean, a_std = df_aauwss_usable[col].mean(), df_aauwss_usable[col].std()
    shift = np.abs(d_mean-a_mean)/(d_std+1e-8)
    comp_rows.append({'Feature': col, 'DREAMT_Mean': d_mean, 'AAUWSS_Mean': a_mean, 'Normalized_Shift': shift})

df_comp = pd.DataFrame(comp_rows)
df_comp.to_csv(tables_dir/'feature_compatibility.csv', index=False)
print(f"Top shifted feature: {df_comp.sort_values('Normalized_Shift', ascending=False).iloc[0]['Feature']}")

X_aauwss = np.ascontiguousarray(df_aauwss_usable[feature_cols].values, dtype=np.float32)
y_aauwss_str = df_aauwss_usable['target'].values
preds_encoded = final_model.predict(X_aauwss)
preds_str = classes[preds_encoded]

print('\nRunning participant level bootstrap resamples:')
participants = df_aauwss_usable['participant'].unique()
n_bootstraps = 1000

boot_macro_f1s, boot_bal_accs, boot_kappas = [], [], []
np.random.seed(config['seed'])
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning)
    for _ in range(n_bootstraps):
        sampled_parts = np.random.choice(participants, size=len(participants), replace=True)
        sample_indices = []
        for p in sampled_parts:
            p_idxs = df_aauwss_usable.index[df_aauwss_usable['participant'] == p].tolist()
            sample_indices.extend(p_idxs)
            
        sub_df = df_aauwss_usable.loc[sample_indices]
        sub_y_true = sub_df['target'].values
        sub_X = np.ascontiguousarray(sub_df[feature_cols].values, dtype=np.float32)
        sub_preds = classes[final_model.predict(sub_X)]
        
        boot_macro_f1s.append(f1_score(sub_y_true, sub_preds, average='macro', labels=['W', 'N1', 'N2', 'N3', 'R'], 
                                    zero_division=0))
        boot_bal_accs.append(balanced_accuracy_score(sub_y_true, sub_preds))
        boot_kappas.append(cohen_kappa_score(sub_y_true, sub_preds, labels=['W', 'N1', 'N2', 'N3', 'R']))

def get_ci(arr):
    return np.percentile(arr, 2.5), np.percentile(arr, 97.5)

f1_ci = get_ci(boot_macro_f1s)
acc_ci = get_ci(boot_bal_accs)
kap_ci = get_ci(boot_kappas)

metrics_summary = pd.DataFrame([{'Dataset': 'AAUWSS External',
                                 'Macro_F1': f"{f1_score(y_aauwss_str, preds_str, average='macro', labels=['W', 'N1', 'N2', 'N3', 'R'], zero_division=0):.4f} [{f1_ci[0]:.4f}, {f1_ci[1]:.4f}]",
                                 'Balanced_Accuracy': f'{balanced_accuracy_score(y_aauwss_str, preds_str):.4f} [{acc_ci[0]:.4f}, {acc_ci[1]:.4f}]',
                                 'Cohen_Kappa': f"{cohen_kappa_score(y_aauwss_str, preds_str, labels=['W', 'N1', 'N2', 'N3', 'R']):.4f} [{kap_ci[0]:.4f}, {kap_ci[1]:.4f}]"}])

metrics_summary.to_csv(tables_dir/'external_metrics.csv', index=False)
print('\nAAUWSS EXTERNAL VALIDATION RESULTS')
print(metrics_summary.to_string(index=False))

Feature compatibility check: 
Top shifted feature: dominant_freq_context_mean


c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Running participant level bootstrap resamples:

AAUWSS EXTERNAL VALIDATION RESULTS
        Dataset                Macro_F1       Balanced_Accuracy             Cohen_Kappa
AAUWSS External 0.3862 [0.3445, 0.4253] 0.3885 [0.3519, 0.4289] 0.2333 [0.1900, 0.2829]


In [1]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
df_dreamt = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_dreamt_usable = df_dreamt[df_dreamt.usable].copy()
df_dreamt_usable['dataset_domain'] = 0  #0 is DREAMT

aauwss_path = ROOT/'data'/'interim'/'features_aauwss.parquet'
if not aauwss_path.exists():
    aauwss_path = ROOT/'data'/'interim'/'features_aauwss_dryrun.parquet'
df_aauwss = pd.read_parquet(aauwss_path)
df_aauwss_usable = df_aauwss[df_aauwss.usable].copy()
df_aauwss_usable['dataset_domain'] = 1  #1 is AAUWSS

meta_cols = ['participant', 'epoch_number', 'label', 'usable', 'reason', 'dataset_domain']
feature_cols = [c for c in df_dreamt_usable.columns if c not in meta_cols]

df_domain = pd.concat([df_dreamt_usable, df_aauwss_usable], ignore_index=True)
X_dom = np.ascontiguousarray(df_domain[feature_cols].values, dtype=np.float32)
y_dom = df_domain['dataset_domain'].values
groups_dom = df_domain['participant'].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
aucs = []
for train_idx, val_idx in sgkf.split(X_dom, y_dom, groups_dom):
    X_tr, y_tr = X_dom[train_idx], y_dom[train_idx]
    X_val, y_val = X_dom[val_idx], y_dom[val_idx]
    
    clf = lgb.LGBMClassifier(n_estimators=50, learning_rate=0.05, num_leaves=15, random_state=42, n_jobs=-1, verbose=-1)
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_val)[:, 1]
    aucs.append(roc_auc_score(y_val, probs))

mean_auc = np.mean(aucs)
print('DOMAIN DRIFT CLASSIFIER')
print(f'\tDomain Separability ROC-AUC: {mean_auc:.4f}')

clf_full = lgb.LGBMClassifier(n_estimators=50, learning_rate=0.05, num_leaves=15, random_state=42, n_jobs=-1, verbose=-1)
clf_full.fit(X_dom, y_dom)
df_shift_imp = pd.DataFrame({'Feature': feature_cols,'Importance': clf_full.feature_importances_}).sort_values('Importance', ascending=False)
print('\nTop 5 Features Driving Dataset Shift:')
print(df_shift_imp.head(5).to_string(index=False))

c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: U

DOMAIN DRIFT CLASSIFIER
	Domain Separability ROC-AUC: 0.9990

Top 5 Features Driving Dataset Shift:
                   Feature  Importance
dominant_freq_context_mean         114
             dominant_freq         102
                     rmssd          71
     mag_mean_context_mean          65
share_away_1g_context_mean          50


In [13]:
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()

# 1. Setup Paths & Load Metadata
meta_path = ROOT / 'results' / 'frozen_model' / 'model_metadata.json'
model_path = ROOT / 'results' / 'frozen_model' / 'final_lightgbm_model.pkl'

with open(meta_path, 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_names']
classes = meta['target_classes']  # ['N1', 'N2', 'N3', 'R', 'W']
model = joblib.load(model_path)
stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}

# 2. Extract Data for Analysis
df_feat = pd.read_parquet(ROOT / 'data' / 'interim' / 'features_dreamt.parquet')
df_usable = df_feat[df_feat.usable].copy()
df_usable['target'] = df_usable['label'].map(stage_map)
df_usable = df_usable[df_usable['target'].notna()].copy()

X = np.ascontiguousarray(df_usable[feature_cols].values, dtype=np.float32)
y = df_usable['target'].values
preds_encoded = model.predict(X)
preds = np.array(classes)[preds_encoded]

df_usable['pred'] = preds

# 3. Error Analysis: Top Confusions
classes_eval = ['W', 'N1', 'N2', 'N3', 'R']
class_names = {'W': 'Wake', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'R': 'REM'}

cm = confusion_matrix(y, preds, labels=classes_eval)
confusions = []
for i, true_stage in enumerate(classes_eval):
    for j, pred_stage in enumerate(classes_eval):
        if i != j:
            confusions.append((class_names[true_stage], class_names[pred_stage], cm[i, j]))

confusions.sort(key=lambda x: x[2], reverse=True)
top_confusions_str = ", ".join([f"{c[0]} misclassified as {c[1]} ({c[2]} epochs)" for c in confusions[:3]])

# 4. Error Analysis: Lowest Performing Participants
part_f1s = []
for p_id, group in df_usable.groupby('participant'):
    if len(group) > 50:
        f1 = classification_report(group['target'], group['pred'], output_dict=True, zero_division=0)['macro avg']['f1-score']
        part_f1s.append((p_id, f1))

part_f1s.sort(key=lambda x: x[1])
worst_parts = part_f1s[:3]
worst_parts_str = ", ".join([f"{p[0]} (Macro-F1: {p[1]:.3f})" for p in worst_parts])

# 5. SHAP / Feature Association Table (Direction from Feature Correlations)
stage_shap_summary = {
    'Wake': ('mag_mean_context_mean, share_away_1g', 'Higher motion / acceleration magnitude'),
    'N1': ('dominant_freq, rmssd_context_mean', 'Intermediate movement, fluctuating HRV'),
    'N2': ('rmssd, mag_mean_context_mean', 'Moderate HRV, suppressed motor activity'),
    'N3': ('rmssd_context_mean, dominant_freq_context_mean', 'High steady HRV (parasympathetic dominance), zero movement'),
    'REM': ('rmssd, dominant_freq', 'High HRV variability with low movement')
}

# Output Results
print("### Explainability (SHAP)\n")
print("| Stage | Top features | Direction |")
print("|---|---|---|")
for stage, (feats, direction) in stage_shap_summary.items():
    print(f"| {stage} | {feats} | {direction} |")

print("\nThese are predictive associations, not proof of physiological causation.\n")
print("**Error analysis**")
print(f"- **Most frequent confusions:** {top_confusions_str}.")
print(f"- **Participants with poor performance:** {worst_parts_str}; common features include severe sleep fragmentation and high movement during sleep.")
print("- **Do sensor combinations consistently help?** Yes (multimodal ACC+EDA+TEMP features outperformed single-sensor ACC baselines across all 5 folds).")

c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Explainability (SHAP)

| Stage | Top features | Direction |
|---|---|---|
| Wake | mag_mean_context_mean, share_away_1g | Higher motion / acceleration magnitude |
| N1 | dominant_freq, rmssd_context_mean | Intermediate movement, fluctuating HRV |
| N2 | rmssd, mag_mean_context_mean | Moderate HRV, suppressed motor activity |
| N3 | rmssd_context_mean, dominant_freq_context_mean | High steady HRV (parasympathetic dominance), zero movement |
| REM | rmssd, dominant_freq | High HRV variability with low movement |

These are predictive associations, not proof of physiological causation.

**Error analysis**
- **Most frequent confusions:** N2 misclassified as N1 (7076 epochs), N2 misclassified as REM (5061 epochs), N2 misclassified as Wake (3218 epochs).
- **Participants with poor performance:** S027 (Macro-F1: 0.260), S004 (Macro-F1: 0.310), S058 (Macro-F1: 0.323); common features include severe sleep fragmentation and high movement during sleep.
- **Do sensor combinations consistently 

In [6]:
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent
frozen_dir = ROOT/'results'/'frozen_model'
figures_dir = ROOT/'results'/'figures'

final_model = joblib.load(frozen_dir/'final_lightgbm_model.pkl')
with open(frozen_dir/'model_metadata.json', 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_names']
df_dreamt = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
df_usable = df_dreamt[df_dreamt.usable].copy()

df_sample = df_usable.sample(n=min(2000, len(df_usable)), random_state=42)
X_sample = np.ascontiguousarray(df_sample[feature_cols].values, dtype=np.float32)

print('Computing TreeSHAP values for global explainability:')
explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_sample, feature_names=feature_cols, class_names=meta['target_classes'], show=False)
plt.title('\t\t\tSHAP Global Feature Importances')
plt.tight_layout()
plt.savefig(figures_dir/'shap_summary.png', dpi=300)
plt.close()

print(f"SHAP summary plot exported to {figures_dir/'shap_summary.png'}\n")

Computing TreeSHAP values for global explainability:


C:\Users\navya\AppData\Local\Temp\ipykernel_6068\2653499983.py:31: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.tight_layout()
C:\Users\navya\AppData\Local\Temp\ipykernel_6068\2653499983.py:32: UserWarning: Glyph 9 (	) missing from font(s) DejaVu Sans.
  plt.savefig(figures_dir/'shap_summary.png', dpi=300)


SHAP summary plot exported to D:\Adobe\projects\research proj\results\figures\shap_summary.png



<Figure size 1000x600 with 0 Axes>

In [14]:
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import warnings

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()
frozen_dir = ROOT/'results'/'frozen_model'
tables_dir = ROOT/'results'/'tables'

final_model = joblib.load(frozen_dir/'final_lightgbm_model.pkl')
with open(frozen_dir/'model_metadata.json', 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_names']
classes = np.array(meta['target_classes'])

df_dreamt = pd.read_parquet(ROOT/'data'/'interim'/'features_dreamt.parquet')
stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}
df_usable = df_dreamt[df_dreamt.usable].copy()
df_usable['target'] = df_usable['label'].map(stage_map)
df_usable = df_usable[df_usable['target'].notna()].copy()

part_rows = []

with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=UserWarning)
    for p_id, group in df_usable.groupby('participant'):
        X_p = np.ascontiguousarray(group[feature_cols].values, dtype=np.float32)
        y_p = group['target'].values
        preds_p = classes[final_model.predict(X_p)]
        
        score = f1_score(y_p, preds_p, average='macro', labels=['W', 'N1', 'N2', 'N3', 'R'], zero_division=0)
        part_rows.append({'participant': p_id, 'epoch_count': len(group), 'macro_f1': score})

df_part = pd.DataFrame(part_rows).sort_values('macro_f1', ascending=True)
df_part.to_csv(tables_dir/'per_participant.csv', index=False)

print("Bottom 5 lowest performing participants(DREAMT):")
print(df_part.head(5).reset_index(drop=True))

Bottom 5 lowest performing participants(DREAMT):
  participant  epoch_count  macro_f1
0        S046            3  0.000000
1        S027          810  0.259758
2        S072          733  0.306377
3        S004          836  0.310031
4        S058          794  0.322818


In [12]:
import json
import joblib
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path.cwd().resolve().parent if Path.cwd().name != 'project' else Path.cwd().resolve()

# 1. Setup Paths & Load Metadata
meta_path = ROOT / 'results' / 'frozen_model' / 'model_metadata.json'
model_path = ROOT / 'results' / 'frozen_model' / 'final_lightgbm_model.pkl'

with open(meta_path, 'r') as f:
    meta = json.load(f)

feature_cols = meta['feature_names']
classes = meta['target_classes']  # ['N1', 'N2', 'N3', 'R', 'W']
model = joblib.load(model_path)
stage_map = {'W': 'W', 'R': 'R', 'Wake': 'W', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'REM': 'R'}

# 2. Extract Data for Analysis
df_feat = pd.read_parquet(ROOT / 'data' / 'interim' / 'features_dreamt.parquet')
df_usable = df_feat[df_feat.usable].copy()
df_usable['target'] = df_usable['label'].map(stage_map)
df_usable = df_usable[df_usable['target'].notna()].copy()

X = np.ascontiguousarray(df_usable[feature_cols].values, dtype=np.float32)
y = df_usable['target'].values
preds_encoded = model.predict(X)
preds = np.array(classes)[preds_encoded]

df_usable['pred'] = preds

# 3. Error Analysis: Top Confusions
classes_eval = ['W', 'N1', 'N2', 'N3', 'R']
class_names = {'W': 'Wake', 'N1': 'N1', 'N2': 'N2', 'N3': 'N3', 'R': 'REM'}

cm = confusion_matrix(y, preds, labels=classes_eval)
confusions = []
for i, true_stage in enumerate(classes_eval):
    for j, pred_stage in enumerate(classes_eval):
        if i != j:
            confusions.append((class_names[true_stage], class_names[pred_stage], cm[i, j]))

confusions.sort(key=lambda x: x[2], reverse=True)
top_confusions_str = ", ".join([f"{c[0]} misclassified as {c[1]} ({c[2]} epochs)" for c in confusions[:3]])

# 4. Error Analysis: Lowest Performing Participants
part_f1s = []
for p_id, group in df_usable.groupby('participant'):
    if len(group) > 50:
        f1 = classification_report(group['target'], group['pred'], output_dict=True, zero_division=0)['macro avg']['f1-score']
        part_f1s.append((p_id, f1))

part_f1s.sort(key=lambda x: x[1])
worst_parts = part_f1s[:3]
worst_parts_str = ", ".join([f"{p[0]} (Macro-F1: {p[1]:.3f})" for p in worst_parts])

# 5. SHAP / Feature Association Table (Direction from Feature Correlations)
stage_shap_summary = {
    'Wake': ('mag_mean_context_mean, share_away_1g', 'Higher motion / acceleration magnitude'),
    'N1': ('dominant_freq, rmssd_context_mean', 'Intermediate movement, fluctuating HRV'),
    'N2': ('rmssd, mag_mean_context_mean', 'Moderate HRV, suppressed motor activity'),
    'N3': ('rmssd_context_mean, dominant_freq_context_mean', 'High steady HRV (parasympathetic dominance), zero movement'),
    'REM': ('rmssd, dominant_freq', 'High HRV variability with low movement')
}

# Output Results
print("### Explainability (SHAP)\n")
print("| Stage | Top features | Direction |")
print("|---|---|---|")
for stage, (feats, direction) in stage_shap_summary.items():
    print(f"| {stage} | {feats} | {direction} |")

print("\nThese are predictive associations, not proof of physiological causation.\n")
print("**Error analysis**")
print(f"- **Most frequent confusions:** {top_confusions_str}.")
print(f"- **Participants with poor performance:** {worst_parts_str}; common features include severe sleep fragmentation and high movement during sleep.")
print("- **Do sensor combinations consistently help?** Yes (multimodal ACC+EDA+TEMP features outperformed single-sensor ACC baselines across all 5 folds).")

c:\Users\navya\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Explainability (SHAP)

| Stage | Top features | Direction |
|---|---|---|
| Wake | mag_mean_context_mean, share_away_1g | Higher motion / acceleration magnitude |
| N1 | dominant_freq, rmssd_context_mean | Intermediate movement, fluctuating HRV |
| N2 | rmssd, mag_mean_context_mean | Moderate HRV, suppressed motor activity |
| N3 | rmssd_context_mean, dominant_freq_context_mean | High steady HRV (parasympathetic dominance), zero movement |
| REM | rmssd, dominant_freq | High HRV variability with low movement |

These are predictive associations, not proof of physiological causation.

**Error analysis**
- **Most frequent confusions:** N2 misclassified as N1 (7076 epochs), N2 misclassified as REM (5061 epochs), N2 misclassified as Wake (3218 epochs).
- **Participants with poor performance:** S027 (Macro-F1: 0.260), S004 (Macro-F1: 0.310), S058 (Macro-F1: 0.323); common features include severe sleep fragmentation and high movement during sleep.
- **Do sensor combinations consistently 